# EnSF Heat Equation: Run Experiments

Runs the Ensemble Score Filter (EnSF) on the parametric heat equation
under three observation densities (100%, 25%, 5%) and saves results to HDF5.

**Problem:** Diffusion on $\Omega = (-1,1)^2$ with Gaussian permeability field
$K(x,y;\theta)$ and nonlinear arctan observation model $Y_t = \arctan(X_t) + \varepsilon_t$.

**Solver:** FEniCSx P1 finite elements, backward Euler time stepping.  
Adapted from: Dokken, J.S. *FEniCSx Tutorial*, Chapter 2 — The Heat Equation.  
https://jsdokken.com/dolfinx-tutorial/chapter2/heat_equation.html

**Filter:** EnSF via reverse-time SDE.  
Reference: Bao et al. (2022), https://arxiv.org/pdf/2309.00983

---
**Outputs** (saved to `../data/`):
- `heat_obs100.h5`
- `heat_obs25.h5`
- `heat_obs5.h5`

In [1]:
import numpy as np
import h5py
from pathlib import Path
from tqdm import tqdm
from matplotlib.tri import Triangulation

from solver import HeatEquationProblem, create_mesh
from ensf import reverse_SDE, make_score_likelihood_arctan

## 1. Mesh

In [2]:
dx = 5e-2
mesh, subdomains, boundaries = create_mesh(dx)

mesh_coords = mesh.geometry.x.copy()
mesh.topology.create_connectivity(mesh.topology.dim, 0)
mesh_cells = mesh.topology.connectivity(mesh.topology.dim, 0).array.copy()

state_dim = mesh_coords.shape[0]
print(f"Mesh: {state_dim} nodes, dx = {dx}")

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 30%] Meshing curve 2 (Line)
Info    : [ 50%] Meshing curve 3 (Line)
Info    : [ 80%] Meshing curve 4 (Line)
Info    : Done meshing 1D (Wall 0.000407412s, CPU 0.00019s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Transfinite)
Info    : Done meshing 2D (Wall 0.00239646s, CPU 0.001407s)
Info    : 1681 nodes 1764 elements
Mesh: 1681 nodes, dx = 0.05


## 2. Experiment parameters

In [3]:
# Time discretization
T             = 2.0    # final time
N             = 100    # assimilation steps
TRUTH_SUBSTEPS = 4     # fine steps per assimilation step

dt_forecast = T / N
dt_truth    = dt_forecast / TRUTH_SUBSTEPS

# True parameter vector theta = [mu_x, mu_y, var_x, var_y]
THETA_TRUE = np.array([0.0, 0.0, 1.0, 1.0])

# Observation model
OBS_SIGMA = 0.15

# EnSF hyperparameters
ENSEMBLE_SIZE = 50
EPS_ALPHA     = 0.05
EULER_STEPS   = 300
PARAM_NOISE   = 0.5    # std of parameter perturbation per particle

# Initial ensemble offset (biased initialization to test filter correction)
INIT_MEAN = 0.5
INIT_STD  = 1.0

SEED = 42
rng  = np.random.default_rng(SEED)

## 3. Generate ground truth trajectory

In [4]:
problem_truth = HeatEquationProblem(mesh, dt=dt_truth, T=T)
truth_solutions, _ = problem_truth.solve_trajectory(THETA_TRUE)

X_true_fine = np.zeros((len(truth_solutions) + 1, state_dim))
for k, uh in enumerate(truth_solutions, start=1):
    X_true_fine[k] = uh.x.array[:]

# Subsample to assimilation times
XTrue = X_true_fine[::TRUTH_SUBSTEPS].copy()
assert XTrue.shape[0] == N + 1

# Observations
Y = np.arctan(XTrue) + OBS_SIGMA * rng.standard_normal(XTrue.shape)
t_vec = np.linspace(dt_forecast, T, N)

print(f"Truth shape: {XTrue.shape}, Observations shape: {Y.shape}")

Truth shape: (101, 1681), Observations shape: (101, 1681)


## 4. Filtering loop (single experiment)

The `run_experiment` function below runs one EnSF experiment for a given observation fraction.

In [5]:
def run_experiment(obs_fraction: float, save_tag: str, save_dir: Path):
    """
    Run one EnSF filtering experiment and save results to HDF5.

    Parameters
    ----------
    obs_fraction : float
        Fraction of state DOFs observed, in (0, 1].
    save_tag : str
        Base filename (without extension) for the HDF5 output.
    save_dir : Path
        Directory where HDF5 file is written.
    """
    rng_exp = np.random.default_rng(SEED)

    # Observation mask
    if obs_fraction >= 1.0:
        observed_mask = np.ones(state_dim, dtype=bool)
    else:
        n_obs = max(1, int(round(obs_fraction * state_dim)))
        obs_idx = rng_exp.choice(state_dim, n_obs, replace=False)
        observed_mask = np.zeros(state_dim, dtype=bool)
        observed_mask[obs_idx] = True

    # Initial ensemble (biased)
    state_particles = INIT_MEAN + INIT_STD * rng_exp.standard_normal((ENSEMBLE_SIZE, state_dim))

    # Storage
    step_rmse      = np.zeros(N)
    obs_save       = np.full((N, state_dim), np.nan)
    EnSF_est_save  = np.zeros((N + 1, state_dim))
    EnSF_est_save[0] = np.mean(state_particles, axis=0)

    # Reuse one forecast solver
    problem_forecast = HeatEquationProblem(mesh, dt=dt_forecast, T=dt_forecast)

    for i in tqdm(range(1, N + 1), desc=f"{save_tag} (obs={obs_fraction:.0%})"):
        t_np1 = i * dt_forecast

        # --- Prediction step ---
        for j in range(ENSEMBLE_SIZE):
            problem_forecast.set_initial_condition_from_array(state_particles[j])
            theta_j = THETA_TRUE + PARAM_NOISE * rng_exp.standard_normal(4)
            theta_j[2:] = np.maximum(theta_j[2:], 1e-6)  # keep variances positive
            uh = problem_forecast.solve_one_step(theta_j, t_np1)
            state_particles[j] = uh.x.array[:]

        # --- Analysis step ---
        obs = Y[i].copy()
        obs_save[i - 1, observed_mask] = obs[observed_mask]

        score_fn = make_score_likelihood_arctan(obs, observed_mask, OBS_SIGMA)
        state_particles = reverse_SDE(
            x0=state_particles,
            ensemble_size=ENSEMBLE_SIZE,
            state_dim=state_dim,
            eps_alpha=EPS_ALPHA,
            score_likelihood=score_fn,
            time_steps=EULER_STEPS,
        )

        EnSF_est = np.mean(state_particles, axis=0)
        EnSF_est_save[i] = EnSF_est
        step_rmse[i - 1] = np.sqrt(np.mean((XTrue[i] - EnSF_est) ** 2))

    # --- Save HDF5 ---
    save_dir.mkdir(parents=True, exist_ok=True)
    h5_path = save_dir / f"{save_tag}.h5"

    with h5py.File(h5_path, "w") as f:
        # Experiment metadata
        f.attrs["T"]              = T
        f.attrs["N"]              = N
        f.attrs["dt_forecast"]    = dt_forecast
        f.attrs["dt_truth"]       = dt_truth
        f.attrs["truth_substeps"] = TRUTH_SUBSTEPS
        f.attrs["obs_sigma"]      = OBS_SIGMA
        f.attrs["obs_fraction"]   = obs_fraction
        f.attrs["ensemble_size"]  = ENSEMBLE_SIZE
        f.attrs["euler_steps"]    = EULER_STEPS
        f.attrs["eps_alpha"]      = EPS_ALPHA
        f.attrs["param_noise"]    = PARAM_NOISE
        f.attrs["seed"]           = SEED

        # Mesh
        f.create_dataset("mesh_coords",   data=mesh_coords)
        f.create_dataset("mesh_cells",    data=mesh_cells)

        # Simulation data
        f.create_dataset("time",          data=t_vec)
        f.create_dataset("theta_true",    data=THETA_TRUE)
        f.create_dataset("observed_mask", data=observed_mask.astype(np.uint8))
        f.create_dataset("XTrue",         data=XTrue)
        f.create_dataset("Y",             data=Y)
        f.create_dataset("obs_save",      data=obs_save)
        f.create_dataset("EnSF_est",      data=EnSF_est_save)
        f.create_dataset("step_rmse",     data=step_rmse)

    print(f"Saved → {h5_path}")
    return h5_path

## 5. Run all three observation scenarios

In [6]:
data_dir = Path("../data")

run_experiment(obs_fraction=1.00, save_tag="heat_obs100", save_dir=data_dir)
run_experiment(obs_fraction=0.25, save_tag="heat_obs25",  save_dir=data_dir)
run_experiment(obs_fraction=0.05, save_tag="heat_obs5",   save_dir=data_dir)

heat_obs100 (obs=100%): 100%|█████████████████| 100/100 [03:31<00:00,  2.11s/it]


Saved → ../data/heat_obs100.h5


heat_obs25 (obs=25%): 100%|███████████████████| 100/100 [02:55<00:00,  1.76s/it]


Saved → ../data/heat_obs25.h5


heat_obs5 (obs=5%): 100%|█████████████████████| 100/100 [02:46<00:00,  1.67s/it]

Saved → ../data/heat_obs5.h5


PosixPath('../data/heat_obs5.h5')